In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)
df = df[['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']]

# Q1: coluna com valores faltando
print("Q1:", df.isnull().sum()[df.isnull().sum() > 0])

# Q2: mediana do horsepower
print("Q2:", df['horsepower'].median())

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

def split_data(df, seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    idx = np.arange(n)
    np.random.seed(seed)
    np.random.shuffle(idx)
    df_shuf = df.iloc[idx].reset_index(drop=True)
    return df_shuf.iloc[:n_train], df_shuf.iloc[n_train:n_train+n_val], df_shuf.iloc[n_train+n_val:]

# Q3: preencher com 0 vs média
for strategy in ["zero", "mean"]:
    train, val, test = split_data(df, seed=42)
    fill_value = 0 if strategy == "zero" else train['horsepower'].mean()
    train_f = train.copy(); val_f = val.copy()
    train_f['horsepower'] = train_f['horsepower'].fillna(fill_value)
    val_f['horsepower'] = val_f['horsepower'].fillna(fill_value)

    y_train = train_f['fuel_efficiency_mpg'].values
    y_val = val_f['fuel_efficiency_mpg'].values
    X_train = train_f.drop(columns=['fuel_efficiency_mpg']).values
    X_val = val_f.drop(columns=['fuel_efficiency_mpg']).values

    w0, w = train_linear_regression_reg(X_train, y_train, r=0)
    y_pred = w0 + X_val.dot(w)
    print(f"Q3 ({strategy}):", round(rmse(y_val, y_pred), 3))

# Q4: melhor r (preenchendo com 0)
train, val, test = split_data(df, seed=42)
train_f = train.fillna(0); val_f = val.fillna(0)
y_train = train_f['fuel_efficiency_mpg'].values
y_val = val_f['fuel_efficiency_mpg'].values
X_train = train_f.drop(columns=['fuel_efficiency_mpg']).values
X_val = val_f.drop(columns=['fuel_efficiency_mpg']).values

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w0 + X_val.dot(w)
    print(f"Q4 r={r}:", round(rmse(y_val, y_pred), 4))

# Q5: desvio padrão do RMSE entre seeds 0-9
scores = []
for seed in range(10):
    train, val, test = split_data(df, seed=seed)
    train_f = train.fillna(0); val_f = val.fillna(0)
    y_train = train_f['fuel_efficiency_mpg'].values
    y_val = val_f['fuel_efficiency_mpg'].values
    X_train = train_f.drop(columns=['fuel_efficiency_mpg']).values
    X_val = val_f.drop(columns=['fuel_efficiency_mpg']).values
    w0, w = train_linear_regression_reg(X_train, y_train, r=0)
    y_pred = w0 + X_val.dot(w)
    scores.append(rmse(y_val, y_pred))
print("Q5:", round(np.std(scores), 3))

# Q6: treino+val juntos, seed=9, r=0.001, avaliar no teste
train, val, test = split_data(df, seed=9)
full_train = pd.concat([train, val]).fillna(0)
test_f = test.fillna(0)
y_full = full_train['fuel_efficiency_mpg'].values
y_test = test_f['fuel_efficiency_mpg'].values
X_full = full_train.drop(columns=['fuel_efficiency_mpg']).values
X_test = test_f.drop(columns=['fuel_efficiency_mpg']).values
w0, w = train_linear_regression_reg(X_full, y_full, r=0.001)
y_pred = w0 + X_test.dot(w)
print("Q6:", round(rmse(y_test, y_pred), 3))

Q1: horsepower    877
dtype: int64
Q2: 254.0
Q3 (zero): 2.205
Q3 (mean): 2.202
Q4 r=0: 2.2053
Q4 r=0.01: 2.2058
Q4 r=0.1: 2.2241
Q4 r=1: 2.3492
Q4 r=5: 2.4094
Q4 r=10: 2.4195
Q4 r=100: 2.4292
Q5: 0.029
Q6: 2.236
